In [2]:
gem_path = "/home/abigaylmontantearenas/Documents/proyecto_tesis/04_resultados/gem"
output_dir = "/home/abigaylmontantearenas/Documents/proyecto_tesis/04_resultados"

FBA

In [ ]:
import os
import cobra
import pandas as pd

gem_path = "/home/abigaylmontantearenas/Documents/proyecto_tesis/04_resultados/gem"
# Archivo general de crecimiento
output_csv_general = "/home/abigaylmontantearenas/Documents/proyecto_tesis/04_resultados/resumen_crecimiento_modelos.csv"
# Carpeta donde guardaremos los reportes de flujos de cada modelo
output_dir_summary = "/home/abigaylmontantearenas/Documents/proyecto_tesis/04_resultados/resumenes_flujos"

# Crear la carpeta de flujos si no existe para mantener el orden
os.makedirs(output_dir_summary, exist_ok=True)

# Lista para almacenar los resultados generales
datos_modelos = []

# Recorrer los archivos en la carpeta
for file in os.listdir(gem_path):
    if file.endswith(".xml"):
        path_completo = os.path.join(gem_path, file)
        nombre_base = os.path.splitext(file)[0] # Nombre del modelo sin el .xml
        
        try:
            # Leer el modelo
            model = cobra.io.read_sbml_model(path_completo)
            
            # Ejecutar FBA (Obligatorio antes de pedir el summary)
            solution = model.optimize()
            
            if solution.status == 'optimal':
                growth_rate = solution.objective_value
                mensaje = "Funcional y con crecimiento" if growth_rate > 1e-6 else "Funcional pero NO crece"
                
                # --- AQUÍ EXTRAEMOS EL SUMMARY EN CSV ---
                # Convertimos el resumen impreso a un DataFrame de Pandas usando .to_frame()
                df_summary = model.summary().to_frame()
                
                # Guardamos el CSV específico de este modelo
                csv_modelo_path = os.path.join(output_dir_summary, f"summary_{nombre_base}.csv")
                df_summary.to_csv(csv_modelo_path, index=True, encoding='utf-8')
                # ----------------------------------------
                
                datos_modelos.append({
                    "Modelo": file,
                    "Estado": solution.status,
                    "Tasa_Crecimiento_h_1": round(growth_rate, 4),
                    "Mensaje": mensaje
                })
            else:
                datos_modelos.append({
                    "Modelo": file,
                    "Estado": solution.status,
                    "Tasa_Crecimiento_h_1": 0.0,
                    "Mensaje": "Modelo matemáticamente infactible (No se generó summary)"
                })
                
        except Exception as e:
            datos_modelos.append({
                "Modelo": file,
                "Estado": "Error de lectura/COBRA",
                "Tasa_Crecimiento_h_1": None,
                "Mensaje": str(e).replace('\n', ' ')
            })

# Crear el DataFrame general y exportarlo
df_general = pd.DataFrame(datos_modelos)
df_general.to_csv(output_csv_general, index=False, encoding='utf-8')

print(f"\n[ÉXITO] Proceso terminado.")
print(f"1. Reporte general guardado en: {output_csv_general}")
print(f"2. Los CSVs con los flujos (In/Out) de cada modelo están en: {output_dir_summary}/")

Evaluar variables

In [1]:
import os
import cobra
import pandas as pd
def variables_totales(gem_path, evaluacion="evaluacion"):
    csv_output_path = os.path.join("/home/abigaylmontantearenas/Documents/proyecto_tesis/04_resultados", evaluacion)
    # Asegurar que la carpeta de destino exista
    os.makedirs(csv_output_path, exist_ok=True)

    rows = []  # Aquí se guardarán todos los resultados

    # Buscar directamente los archivos en la carpeta provista
    for file_name in os.listdir(gem_path):
        if file_name.endswith(".xml"):
            
            if '-draft' in file_name:
                continue

            model_id = file_name.replace('.xml', '')
            model_path = os.path.join(gem_path, file_name)

            try:
                cobra_model = cobra.io.read_sbml_model(model_path)

                R = len(cobra_model.reactions)
                M = len(cobra_model.metabolites)
                G = len(cobra_model.genes)

                print(f"{model_id}: R={R} M={M} G={G}")

                rows.append({
                    "Model_ID": model_id,
                    "Total_Reactions": R,
                    "Total_Metabolites": M,
                    "Total_Genes": G
                })

            except Exception as e:
                print(f"ERROR cargando {model_id}: {e}")

    df = pd.DataFrame(rows)

    output_file = os.path.join(csv_output_path, "variables_totales.csv")
    df.to_csv(output_file, index=False)

    print(f"\nCSV guardado: {output_file}")
    print(df)

In [ ]:
variables_totales(gem_path, output_csv)

In [ ]:
import os
import sys

# agregar ruta
sys.path.append('/home/abigaylmontantearenas/Documents/proyecto_tesis/02_data/PhyloMint')

#from lib import BuildGraphNetX, CalculateIndexes

# input_dir = '/home/abigaylmontantearenas/Documents/proyecto_tesis/02_data/rizo/carveme/prokka'
output_path = '/home/abigaylmontantearenas/Documents/proyecto_tesis/phylomint'

# maxcc = 5

# # obtener archivos xml
# files = [f for f in os.listdir(input_dir) if f.endswith('.xml')]

# # guardar datos
# ConfidenceDic = {}
# nonSeedSetDic = {}

# # calcular seed sets
# for f in files:
#     path = os.path.join(input_dir, f)
#     name = f.replace('.xml', '')

#     print(f'Procesando: {name}')

#     DG = BuildGraphNetX.buildDG(path)
#     conf, seed, nonseed = BuildGraphNetX.getSeedSet(DG, maxComponentSize=maxcc)

#     ConfidenceDic[name] = conf
#     nonSeedSetDic[name] = nonseed

# # calcular índices
# with open(output_file, 'w') as out:
#     out.write('A\tB\tCompetition\tComplementarity\n')

#     for A in files:
#         for B in files:
#             A_name = A.replace('.xml', '')
#             B_name = B.replace('.xml', '')

#             comp = CalculateIndexes.MetabolicCompetitionIdx(
#                 ConfidenceDic[A_name], ConfidenceDic[B_name]
#             )

#             coop = CalculateIndexes.MetabolicCooperationIdx(
#                 ConfidenceDic[A_name],
#                 ConfidenceDic[B_name],
#                 nonSeedSetDic[B_name]
#             )

#             out.write(f'{A_name}\t{B_name}\t{comp}\t{coop}\n')

import os
from lib import BuildGraphNetX

sbml_path = '/home/abigaylmontantearenas/Documents/proyecto_tesis/02_data/rizo/carveme/prokka/ST00042_prokka_carveme_lb.xml'

# construir grafo
DG = BuildGraphNetX.buildDG(sbml_path)

# obtener seed set
seed_set = BuildGraphNetX.getSeedSet(DG, maxComponentSize=5)

print("Seed set:", seed_set)
print("Tamaño:", len(seed_set))


